In [ ]:
import os, pickle, json
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import open_clip
import hnswlib
from ultralytics import YOLO
import random

INDEX_DIR = './index'
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'

with open(os.path.join(INDEX_DIR, 'config.json')) as f:
    cfg = json.load(f)

EMBED_DIM = cfg['embed_dim']
K_VALUES  = [5, 10, 15]
print("Config:", json.dumps(cfg, indent=2))

In [ ]:
def recall_at_k(retrieved_ids: list, relevant_ids: set, k: int) -> float:
    return float(any(r in relevant_ids for r in retrieved_ids[:k]))


def ndcg_at_k(retrieved_ids: list, relevant_ids: set, k: int) -> float:
    gains = [1.0 if r in relevant_ids else 0.0 for r in retrieved_ids[:k]]
    dcg   = sum(g / np.log2(i + 2) for i, g in enumerate(gains))

    n_rel  = min(len(relevant_ids), k)
    ideal  = sum(1.0 / np.log2(i + 2) for i in range(n_rel))
    return dcg / ideal if ideal > 0 else 0.0


def average_precision_at_k(retrieved_ids: list, relevant_ids: set, k: int) -> float:
    hits, precision_sum = 0, 0.0
    for i, r in enumerate(retrieved_ids[:k]):
        if r in relevant_ids:
            hits += 1
            precision_sum += hits / (i + 1)
    return precision_sum / min(len(relevant_ids), k) if relevant_ids else 0.0


def compute_metrics(all_retrieved: list, all_relevant: list, k_values=K_VALUES):
    results = {}
    for k in k_values:
        recalls = [recall_at_k(ret, rel, k)
                   for ret, rel in zip(all_retrieved, all_relevant)]
        ndcgs   = [ndcg_at_k(ret, rel, k)
                   for ret, rel in zip(all_retrieved, all_relevant)]
        maps    = [average_precision_at_k(ret, rel, k)
                   for ret, rel in zip(all_retrieved, all_relevant)]
        results[k] = {
            'Recall@K':  (np.mean(recalls), np.std(recalls)),
            'NDCG@K':    (np.mean(ndcgs),   np.std(ndcgs)),
            'mAP@K':     (np.mean(maps),    np.std(maps)),
        }
    return results


def print_metrics(results: dict, label: str = ''):
    print(f"  {label}")
    print(f"{'Metric':<15} {'K=5':>12} {'K=10':>12} {'K=15':>12}")
    print('-'*55)
    for metric in ['Recall@K', 'NDCG@K', 'mAP@K']:
        row = ''
        for k in K_VALUES:
            mean, std = results[k][metric]
            row += f"  {mean:.4f}±{std:.4f}"
        print(f"{metric:<15}{row}\n")



## 3. Load Gallery Index & Metadata

In [ ]:
with open(os.path.join(INDEX_DIR, 'gallery_meta.pkl'), 'rb') as f:
    meta = pickle.load(f)
gallery_item_ids = meta['item_ids']
gallery_captions = meta['captions']
gallery_df       = pd.read_csv('data/gallery.csv')
query_df         = pd.read_csv('data/query.csv')

gallery_id_to_indices = {}
for idx, iid in enumerate(gallery_item_ids):
    gallery_id_to_indices.setdefault(iid, set()).add(iid)

print(f"Query images : {len(query_df):,}")
print(f"Gallery items: {len(gallery_item_ids):,}")

In [ ]:
def load_clip(finetuned_path=None):
    model, _, preprocess = open_clip.create_model_and_transforms(
        'ViT-B-32', pretrained='openai')
    if finetuned_path and os.path.exists(finetuned_path):
        model.load_state_dict(torch.load(finetuned_path, map_location=DEVICE))
        print(f"  Loaded fine-tuned weights: {finetuned_path}")
    model = model.to(DEVICE).eval()
    return model, preprocess


@torch.no_grad()
def embed_query(img, preprocess, clip_model):
    t = preprocess(img).unsqueeze(0).to(DEVICE)
    e = clip_model.encode_image(t)
    e = e / e.norm(dim=-1, keepdim=True)
    return e.cpu().float().numpy()


def build_gallery_embeddings(gallery_df, clip_model, preprocess, yolo_model, alpha, blip2_caption_dict=None):
    embeddings = []
    yolo = YOLO(yolo_model)

    for _, row in tqdm(gallery_df.iterrows(), total=len(gallery_df), desc='Building gallery embeddings', leave=False):
        try:
            img  = Image.open(row['full_path']).convert('RGB')
            res  = yolo(img, verbose=False)
            if res[0].boxes is not None and len(res[0].boxes) > 0:
                b = res[0].boxes.xyxy[int(res[0].boxes.conf.argmax())].int().tolist()
                img = img.crop(b)
            v_emb = embed_query(img, preprocess, clip_model)[0]

            if blip2_caption_dict is not None and alpha < 1.0:
                cap   = blip2_caption_dict.get(row['image_path'], '')
                clip_tokenizer = open_clip.get_tokenizer('ViT-B-32')
                tok   = clip_tokenizer([cap]).to(DEVICE)
                t_emb = clip_model.encode_text(tok)
                t_emb = (t_emb / t_emb.norm(dim=-1, keepdim=True)).cpu().float().numpy()[0]
                fused = alpha * v_emb + (1 - alpha) * t_emb
                fused = fused / (np.linalg.norm(fused) + 1e-8)
                embeddings.append(fused)
            else:
                embeddings.append(v_emb)
        except Exception:
            embeddings.append(np.zeros(EMBED_DIM, dtype=np.float32))

    return np.stack(embeddings).astype(np.float32)


def run_evaluation(gallery_embs, query_df, gallery_item_ids, clip_model, preprocess, yolo, k_values=K_VALUES, top_k=15):
    n = len(gallery_embs)
    idx = hnswlib.Index(space='cosine', dim=gallery_embs.shape[1])
    idx.init_index(max_elements=n, ef_construction=200, M=16)
    idx.add_items(gallery_embs, np.arange(n))
    idx.set_ef(200)

    all_retrieved, all_relevant = [], []

    for _, row in tqdm(query_df.iterrows(), total=len(query_df), desc='Evaluating queries', leave=False):
        try:
            img  = Image.open(row['full_path']).convert('RGB')
            res  = yolo(img, verbose=False)
            if res[0].boxes is not None and len(res[0].boxes) > 0:
                b = res[0].boxes.xyxy[int(res[0].boxes.conf.argmax())].int().tolist()
                img = img.crop(b)
            q_emb = embed_query(img, preprocess, clip_model)
            labels, _ = idx.knn_query(q_emb, k=top_k)
            retrieved  = [gallery_item_ids[l] for l in labels[0]]
        except Exception:
            retrieved = []

        relevant = {row['item_id']}
        all_retrieved.append(retrieved)
        all_relevant.append(relevant)

    return compute_metrics(all_retrieved, all_relevant, k_values)

In [ ]:
with open(os.path.join(INDEX_DIR, 'gallery_meta.pkl'), 'rb') as f:
    meta = pickle.load(f)

caption_dict = dict(zip(gallery_df['image_path'].tolist(), meta['captions']))

yolo_model = YOLO(cfg.get('yolo_model', 'yolov8n.pt'))

ablation_results = {}
SEEDS = [42, 7, 13, 99]  

print("\n[A] Vision-only CLIP (α=1, frozen)")
seed_metrics = []
for seed in SEEDS:
    torch.manual_seed(seed)
    clip_A, pre_A = load_clip(finetuned_path=None)
    # Build embeddings (vision only)
    gallery_embs_A = build_gallery_embeddings(
        gallery_df, clip_A, pre_A, cfg.get('yolo_model','yolov8n.pt'),
        alpha=1.0, blip2_caption_dict=None)
    metrics = run_evaluation(gallery_embs_A, query_df, gallery_item_ids,
                              clip_A, pre_A, yolo_model)
    seed_metrics.append(metrics)

ablation_results['A'] = seed_metrics
print_metrics(seed_metrics[0], label='Config A — Vision-only CLIP')

In [ ]:
ALPHA_VALUES_B = [random.random() for _ in range(2)]

for alpha_b in ALPHA_VALUES_B:
    key = f'B_alpha{alpha_b}'
    print(f"\n[B] Frozen CLIP + BLIP-2, α={alpha_b}")
    seed_metrics = []
    for seed in SEEDS:
        torch.manual_seed(seed)
        clip_B, pre_B = load_clip(finetuned_path=None)
        gallery_embs_B = build_gallery_embeddings(
            gallery_df, clip_B, pre_B, cfg.get('yolo_model','yolov8n.pt'),
            alpha=alpha_b, blip2_caption_dict=caption_dict)
        metrics = run_evaluation(gallery_embs_B, query_df, gallery_item_ids,
                                  clip_B, pre_B, yolo_model)
        seed_metrics.append(metrics)

    ablation_results[key] = seed_metrics
    print_metrics(seed_metrics[0], label=f'Config B — α={alpha_b}')

In [ ]:
FINETUNED_CKPT = 'checkpoints/clip_finetuned_full.pt'
ALPHA_VALUES_C = [random.random() for _ in range(2)]

for alpha_c in ALPHA_VALUES_C:
    key = f'C_alpha{alpha_c}'
    print(f"\n[C] Fine-tuned CLIP + BLIP-2, α={alpha_c}")
    seed_metrics = []
    for seed in SEEDS:
        torch.manual_seed(seed)
        clip_C, pre_C = load_clip(finetuned_path=FINETUNED_CKPT)
        gallery_embs_C = build_gallery_embeddings(
            gallery_df, clip_C, pre_C, cfg.get('yolo_model','yolov8n.pt'),
            alpha=alpha_c, blip2_caption_dict=caption_dict)
        metrics = run_evaluation(gallery_embs_C, query_df, gallery_item_ids,
                                  clip_C, pre_C, yolo_model)
        seed_metrics.append(metrics)

    ablation_results[key] = seed_metrics
    print_metrics(seed_metrics[0], label=f'Config C — α={alpha_c}')

In [ ]:
def aggregate_seeds(seed_metrics_list, k=10, metric='Recall@K'):
    vals = [m[k][metric][0] for m in seed_metrics_list]
    return np.mean(vals), np.std(vals)

configs = list(ablation_results.keys())

rows = []
for cfg_key in configs:
    seed_metrics = ablation_results[cfg_key]
    for k in K_VALUES:
        for metric in ['Recall@K', 'NDCG@K', 'mAP@K']:
            mean, std = aggregate_seeds(seed_metrics, k=k, metric=metric)
            rows.append({'Config': cfg_key, 'K': k, 'Metric': metric,
                         'Mean': mean, 'Std': std})

results_df = pd.DataFrame(rows)
pivot = results_df.pivot_table(
    index=['Config','Metric'], columns='K',
    values=['Mean','Std'], aggfunc='first')
print(pivot.to_string())
results_df.to_csv('data/ablation_results.csv', index=False)

In [ ]:
metric_names = ['Recall@K', 'NDCG@K', 'mAP@K']
colors = plt.cm.Set2(np.linspace(0, 1, len(configs)))

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)

for ax, metric in zip(axes, metric_names):
    sub = results_df[results_df['Metric'] == metric]
    for col_i, cfg_key in enumerate(configs):
        cfg_sub = sub[sub['Config'] == cfg_key]
        means   = cfg_sub.set_index('K')['Mean']
        stds    = cfg_sub.set_index('K')['Std']
        ax.errorbar(K_VALUES, [means[k] for k in K_VALUES],
                    yerr=[stds[k] for k in K_VALUES],
                    label=cfg_key, marker='o', color=colors[col_i],
                    capsize=4, linewidth=1.8)

    ax.set_title(metric, fontweight='bold')
    ax.set_xlabel('K')
    ax.set_xticks(K_VALUES)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

plt.suptitle('Ablation Study — A vs B vs C', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
best_cfg_key = configs[-1]
print(f"Analyzing per-query performance for config: {best_cfg_key}")

hnsw_index = hnswlib.Index(space='cosine', dim=EMBED_DIM)
hnsw_index.load_index(os.path.join(INDEX_DIR, 'hnsw_gallery.bin'),
                       max_elements=cfg['n_gallery'])
hnsw_index.set_ef(200)

clip_best, pre_best = load_clip(FINETUNED_CKPT)
yolo_m = YOLO(cfg.get('yolo_model', 'yolov8n.pt'))

per_query = []
for _, row in tqdm(query_df.iterrows(), total=len(query_df), desc='Per-query eval'):
    try:
        img = Image.open(row['full_path']).convert('RGB')
        res = yolo_m(img, verbose=False)
        if res[0].boxes is not None and len(res[0].boxes) > 0:
            b = res[0].boxes.xyxy[int(res[0].boxes.conf.argmax())].int().tolist()
            img = img.crop(b)
        q_emb = embed_query(img, pre_best, clip_best)
        labels, _ = hnsw_index.knn_query(q_emb, k=10)
        retrieved  = [gallery_item_ids[l] for l in labels[0]]
        r10 = recall_at_k(retrieved, {row['item_id']}, 10)
        per_query.append({'item_id': row['item_id'], 'Recall@10': r10})
    except Exception:
        per_query.append({'item_id': row['item_id'], 'Recall@10': 0.0})

pq_df = pd.DataFrame(per_query)
print(f"\nOverall Recall@10: {pq_df['Recall@10'].mean():.4f}")
print(f"Hard queries (R@10=0): {(pq_df['Recall@10']==0).sum()}")
print(f"Easy queries (R@10=1): {(pq_df['Recall@10']==1).sum()}")

In [ ]:
print("FINAL ABLATION TABLE (mean ± std, 3–4 seeds)")
header = f"{'Config':<18} {'R@5':>12} {'R@10':>12} {'R@15':>12} {'NDCG@10':>12} {'mAP@10':>12}"
print(header)
print("-"*70)

for cfg_key in configs:
    seed_metrics = ablation_results[cfg_key]
    r5  = aggregate_seeds(seed_metrics, k=5,  metric='Recall@K')
    r10 = aggregate_seeds(seed_metrics, k=10, metric='Recall@K')
    r15 = aggregate_seeds(seed_metrics, k=15, metric='Recall@K')
    n10 = aggregate_seeds(seed_metrics, k=10, metric='NDCG@K')
    m10 = aggregate_seeds(seed_metrics, k=10, metric='mAP@K')

    def fmt(mean_std):
        return f"{mean_std[0]:.4f}±{mean_std[1]:.4f}"

    print(f"{cfg_key:<18} {fmt(r5):>12} {fmt(r10):>12} {fmt(r15):>12} "
          f"{fmt(n10):>12} {fmt(m10):>12}")